# Verify `voxelize_edge_gpu` vs `voxel_traverse_edge_dda_gpu`

This notebook compares the two edge voxelization paths exposed by the selected API:

1. Choose whether to JIT compile the current local sources or directly import an already-installed `o_voxel` package.
2. Load the selected Python API.
3. Generate synthetic Gaussian triangle soup directly in the notebook, including a few degenerate triangles.
4. Extract unique edges and compare the OCT and DDA voxelization results using pair-level and voxel-level Jaccard scores.

No local mesh file is loaded.


In [1]:
import os
import sys
import time
import types
import importlib
import importlib.util
from pathlib import Path

import numpy as np
import torch
from torch.utils.cpp_extension import load


In [3]:
ROOT = Path(r'/mnt/nvmefs/Projects/Part Generation/TRELLIS.2-o-voxel-gpu-mod/o-voxel').resolve()
USE_JIT = False
INSTALLED_IMPORT_NAME = 'o_voxel'

print('ROOT =', ROOT)
print('USE_JIT =', USE_JIT)
print('torch =', torch.__version__)
print('cuda available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('cuda device =', torch.cuda.get_device_name(0))


def build_jit_extension():
    sources = [
    'src/hash/hash.cu',
    'src/convert/flexible_dual_grid.cpp',
    'src/convert/volumetic_attr.cpp',
    'src/convert/mesh_to_flexible_dual_grid_gpu/torch_bindings.cu',
    'src/convert/mesh_to_flexible_dual_grid_gpu/flexible_dual_grid_gpu.cu',
    'src/convert/mesh_to_flexible_dual_grid_gpu/intersection_qef.cu',
    'src/convert/mesh_to_flexible_dual_grid_gpu/voxelize_mesh_oct.cu',
    'src/convert/mesh_to_flexible_dual_grid_gpu/voxel_traverse_edge_dda.cu',
    'src/serialize/api.cu',
    'src/serialize/hilbert.cu',
    'src/serialize/z_order.cu',
    'src/io/svo.cpp',
    'src/io/filter_parent.cpp',
    'src/io/filter_neighbor.cpp',
    'src/rasterize/rasterize.cu',
    'src/ext.cpp',
]
    full_sources = [str(ROOT / s) for s in sources]
    missing = [s for s in full_sources if not Path(s).exists()]
    if missing:
        raise FileNotFoundError(f'Missing sources: {missing}')

    build_dir = ROOT / '.verify_build'
    build_dir.mkdir(parents=True, exist_ok=True)

    unique_suffix = f"{os.getpid()}_{time.time_ns()}_{os.urandom(4).hex()}"
    mod_name = f"o_voxel_verify_{unique_suffix}"

    max_jobs = max(1, os.cpu_count() or 1)
    os.environ['MAX_JOBS'] = str(max_jobs)
    print('MAX_JOBS =', os.environ['MAX_JOBS'])
    print('JIT module name =', mod_name)

    ext_mod = load(
        name=mod_name,
        sources=full_sources,
        extra_include_paths=[str(ROOT / 'third_party/eigen')],
        extra_cflags=['-O3', '-std=c++17'],
        extra_cuda_cflags=['-O3', '-std=c++17', '--expt-relaxed-constexpr'],
        with_cuda=True,
        build_directory=str(build_dir),
        verbose=True,
    )
    print('JIT build/link: OK')
    print('jit module path =', ext_mod.__file__)
    return ext_mod


def load_local_flexible_dual_grid(ext_mod):
    pkg = types.ModuleType('o_voxel')
    pkg.__path__ = [str(ROOT / 'o_voxel')]
    pkg._C = ext_mod
    sys.modules['o_voxel'] = pkg
    sys.modules['o_voxel._C'] = ext_mod

    convert_pkg = types.ModuleType('o_voxel.convert')
    convert_pkg.__path__ = [str(ROOT / 'o_voxel' / 'convert')]
    sys.modules['o_voxel.convert'] = convert_pkg

    spec = importlib.util.spec_from_file_location(
        'o_voxel.convert.flexible_dual_grid',
        ROOT / 'o_voxel' / 'convert' / 'flexible_dual_grid.py',
    )
    mod = importlib.util.module_from_spec(spec)
    sys.modules['o_voxel.convert.flexible_dual_grid'] = mod
    spec.loader.exec_module(mod)
    return mod


if USE_JIT:
    ext_mod = build_jit_extension()
    fdg_api = load_local_flexible_dual_grid(ext_mod)
    api_mode = 'jit'
else:
    installed_pkg = importlib.import_module(INSTALLED_IMPORT_NAME)
    ext_mod = installed_pkg._C
    fdg_api = importlib.import_module(f'{INSTALLED_IMPORT_NAME}.convert.flexible_dual_grid')
    api_mode = 'installed'
    print('installed package path =', getattr(installed_pkg, '__file__', '<package>'))
    print('installed extension path =', getattr(ext_mod, '__file__', '<extension>'))
    print('installed API path =', getattr(fdg_api, '__file__', '<module>'))

print('api_mode =', api_mode)
print('ext_mod =', ext_mod)
print('fdg_api =', fdg_api)


ROOT = /mnt/nvmefs/Projects/Part Generation/TRELLIS.2-o-voxel-gpu-mod/o-voxel
USE_JIT = False
torch = 2.6.0+cu124
cuda available = True
cuda device = NVIDIA GeForce RTX 4090
installed package path = /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/o_voxel/__init__.py
installed extension path = /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/o_voxel/_C.cpython-310-x86_64-linux-gnu.so
installed API path = /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/o_voxel/convert/flexible_dual_grid.py
api_mode = installed
ext_mod = <module 'o_voxel._C' from '/home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/o_voxel/_C.cpython-310-x86_64-linux-gnu.so'>
fdg_api = <module 'o_voxel.convert.flexible_dual_grid' from '/home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/o_voxel/convert/flexible_dual_grid.py'>


In [4]:
print('has mesh_to_flexible_dual_grid_cpu =', hasattr(ext_mod, 'mesh_to_flexible_dual_grid_cpu'))
print('has mesh_to_flexible_dual_grid_gpu =', hasattr(ext_mod, 'mesh_to_flexible_dual_grid_gpu'))
print('has intersection_occ =', hasattr(fdg_api, 'intersection_occ'))
print('has intersect_qef =', hasattr(fdg_api, 'intersect_qef'))
print('has voxelize_mesh_gpu =', hasattr(fdg_api, 'voxelize_mesh_gpu'))
print('has voxelize_edge_gpu =', hasattr(fdg_api, 'voxelize_edge_gpu'))
print('has face_qef =', hasattr(fdg_api, 'face_qef'))
print('has voxel_traverse_edge_dda_gpu =', hasattr(fdg_api, 'voxel_traverse_edge_dda_gpu'))
print('has boundary_qef =', hasattr(fdg_api, 'boundary_qef'))


has mesh_to_flexible_dual_grid_cpu = True
has mesh_to_flexible_dual_grid_gpu = True
has intersection_occ = True
has intersect_qef = True
has voxelize_mesh_gpu = True
has voxelize_edge_gpu = True
has face_qef = True
has voxel_traverse_edge_dda_gpu = True
has boundary_qef = True


In [5]:
assert torch.cuda.is_available(), 'CUDA is required for this notebook.'
device = torch.device('cuda:0')
torch.cuda.set_device(device)

GRID = 128
N_VERT = 8000
N_FACE = 24000
EDGE_LIMIT = 60000
CHUNK_STEPS = 256
AABB = torch.tensor([[0.0, 0.0, 0.0], [1.0, 1.0, 1.0]], dtype=torch.float32, device=device)

torch.manual_seed(11)
vertices = (0.5 + 0.18 * torch.randn(N_VERT, 3, device=device, dtype=torch.float32)).clamp_(0.0, 1.0)
vertices[1] = vertices[0]
vertices[3] = vertices[2]

faces = torch.randint(0, N_VERT, (N_FACE, 3), device=device, dtype=torch.int64)
faces[:3] = torch.tensor([[0, 0, 1], [2, 3, 3], [4, 4, 4]], device=device, dtype=torch.int64)
faces = faces.to(torch.int32).contiguous()

e01 = faces[:, [0, 1]]
e12 = faces[:, [1, 2]]
e20 = faces[:, [2, 0]]
edges_all = torch.cat([e01, e12, e20], dim=0)
edges_all = torch.sort(edges_all, dim=1).values
edges_unique = torch.unique(edges_all, dim=0)
num_edges = min(EDGE_LIMIT, int(edges_unique.shape[0]))
perm = torch.randperm(edges_unique.shape[0], device=device)
edges = edges_unique[perm[:num_edges]].contiguous().to(torch.int32)

voxel_size = ((AABB[1] - AABB[0]) / GRID).to(torch.float32).contiguous()
grid_range = torch.tensor([[0, 0, 0], [GRID, GRID, GRID]], device=device, dtype=torch.int32)

print('vertices:', vertices.shape)
print('faces:', faces.shape)
print('edges:', edges.shape)
print('voxel_size:', voxel_size)
print('grid_range:', grid_range)


vertices: torch.Size([8000, 3])
faces: torch.Size([24000, 3])
edges: torch.Size([60000, 2])
voxel_size: tensor([0.0078, 0.0078, 0.0078], device='cuda:0')
grid_range: tensor([[  0,   0,   0],
        [128, 128, 128]], device='cuda:0', dtype=torch.int32)


In [6]:
edge_id_oct, voxel_ijk_oct = fdg_api.voxelize_edge_gpu(
    vertices,
    edges,
    voxel_size=voxel_size,
    grid_range=grid_range,
)
edge_id_dda, voxel_ijk_dda = fdg_api.voxel_traverse_edge_dda_gpu(
    vertices,
    edges,
    voxel_size=voxel_size,
    grid_range=grid_range,
    chunk_steps=CHUNK_STEPS,
)

print('oct pairs:', edge_id_oct.shape[0])
print('dda pairs:', edge_id_dda.shape[0])


oct pairs: 4724738
dda pairs: 4724664


In [7]:
gx, gy, gz = [int(x) for x in (grid_range[1] - grid_range[0]).tolist()]

pair_keys_oct = (
    edge_id_oct.to(torch.int64)
    + edges.shape[0]
    * (
        voxel_ijk_oct[:, 0].to(torch.int64)
        + gx * (voxel_ijk_oct[:, 1].to(torch.int64) + gy * voxel_ijk_oct[:, 2].to(torch.int64))
    )
)
pair_keys_dda = (
    edge_id_dda.to(torch.int64)
    + edges.shape[0]
    * (
        voxel_ijk_dda[:, 0].to(torch.int64)
        + gx * (voxel_ijk_dda[:, 1].to(torch.int64) + gy * voxel_ijk_dda[:, 2].to(torch.int64))
    )
)

pair_unique_oct = torch.unique(pair_keys_oct)
pair_unique_dda = torch.unique(pair_keys_dda)
num_pair_oct = int(pair_unique_oct.shape[0])
num_pair_dda = int(pair_unique_dda.shape[0])
num_pair_oct_only = int((~torch.isin(pair_unique_oct, pair_unique_dda)).sum())
num_pair_dda_only = int((~torch.isin(pair_unique_dda, pair_unique_oct)).sum())
num_pair_intersection = num_pair_oct - num_pair_oct_only
num_pair_union = num_pair_intersection + num_pair_oct_only + num_pair_dda_only
pair_jaccard = float(num_pair_intersection / num_pair_union) if num_pair_union > 0 else 1.0

print('pair oct:', num_pair_oct)
print('pair dda:', num_pair_dda)
print('pair intersection:', num_pair_intersection)
print('pair only oct:', num_pair_oct_only)
print('pair only dda:', num_pair_dda_only)
print('pair union:', num_pair_union)
print('pair jaccard:', pair_jaccard)


pair oct: 4724738
pair dda: 4724664
pair intersection: 4724664
pair only oct: 74
pair only dda: 0
pair union: 4724738
pair jaccard: 0.9999843377558714


In [8]:
vox_unique_oct = torch.unique(voxel_ijk_oct.to(torch.int32), dim=0)
vox_unique_dda = torch.unique(voxel_ijk_dda.to(torch.int32), dim=0)

vox_keys_oct = (
    vox_unique_oct[:, 0].to(torch.int64)
    + gx * (vox_unique_oct[:, 1].to(torch.int64) + gy * vox_unique_oct[:, 2].to(torch.int64))
)
vox_keys_dda = (
    vox_unique_dda[:, 0].to(torch.int64)
    + gx * (vox_unique_dda[:, 1].to(torch.int64) + gy * vox_unique_dda[:, 2].to(torch.int64))
)

vox_oct_only_mask = ~torch.isin(vox_keys_oct, vox_keys_dda)
vox_dda_only_mask = ~torch.isin(vox_keys_dda, vox_keys_oct)
vox_oct_only = vox_unique_oct[vox_oct_only_mask]
vox_dda_only = vox_unique_dda[vox_dda_only_mask]

num_vox_oct = int(vox_unique_oct.shape[0])
num_vox_dda = int(vox_unique_dda.shape[0])
num_vox_oct_only = int(vox_oct_only.shape[0])
num_vox_dda_only = int(vox_dda_only.shape[0])
num_vox_intersection = num_vox_oct - num_vox_oct_only
num_vox_union = num_vox_intersection + num_vox_oct_only + num_vox_dda_only
vox_jaccard = float(num_vox_intersection / num_vox_union) if num_vox_union > 0 else 1.0

print('voxel oct:', num_vox_oct)
print('voxel dda:', num_vox_dda)
print('voxel intersection:', num_vox_intersection)
print('voxel only oct:', num_vox_oct_only)
print('voxel only dda:', num_vox_dda_only)
print('voxel union:', num_vox_union)
print('voxel jaccard:', vox_jaccard)


voxel oct: 731279
voxel dda: 731216
voxel intersection: 731216
voxel only oct: 63
voxel only dda: 0
voxel union: 731279
voxel jaccard: 0.9999138495704102


In [9]:
print('sample voxel only-oct:')
print(vox_oct_only[:50].detach().cpu())
print('sample voxel only-dda:')
print(vox_dda_only[:50].detach().cpu())


sample voxel only-oct:
tensor([[ 22, 127,  38],
        [ 22, 127,  39],
        [ 23, 127,  39],
        [ 24, 127,  39],
        [ 24, 127,  40],
        [ 25, 127,  40],
        [ 26, 127,  40],
        [ 26, 127,  41],
        [ 27, 127,  41],
        [ 28, 127,  41],
        [ 28, 127,  42],
        [ 29, 127,  42],
        [ 30, 127,  42],
        [ 30, 127,  43],
        [ 31, 127,  43],
        [ 32, 127,  43],
        [ 32, 127,  44],
        [ 33, 127,  44],
        [ 33, 127,  45],
        [ 34, 127,  45],
        [ 35, 127,  45],
        [ 35, 127,  46],
        [ 36, 127,  46],
        [ 37, 127,  46],
        [ 37, 127,  47],
        [ 38, 127,  47],
        [ 39, 127,  47],
        [ 39, 127,  48],
        [ 40, 127,  48],
        [ 41, 127,  48],
        [ 41, 127,  49],
        [ 42, 127,  49],
        [ 43, 127,  49],
        [ 43, 127,  50],
        [ 44, 127,  50],
        [ 44, 127,  51],
        [ 45, 127,  51],
        [ 46, 127,  51],
        [ 46, 127,  52],
  